# Notebook 2: Electrode-Wise and Region-Wise Model Evaluation

This notebook conducts a comprehensive biometric evaluation across individual electrodes and brain regions using all available EEG files:
1. **Strict Spectral Feature Extraction**: Extracts 20 spectral-domain features per electrode channel with zero non-spectral features.
2. **Standardized Epoch Cap**: Enforces the project-wide common cap of 45 epochs across all files.
3. **80:20 Impostor Testing Protocol**: Splits each genuine user's data 80:20, strictly isolating the 20% genuine test samples and pairing them with an identical number of impostor samples drawn from other users' test splits (zero data leakage).
4. **Comprehensive Model Evaluation**: Runs all 8 advanced models (4 ML and 4 DL architectures) across:
   - All 5 predefined brain regions (Frontal, Central, Temporal, Parietal, Occipital) plus the full 58-channel montage (ALL).
   - 10 representative individual electrode sites across all lobes (Fp1, Fz, FCz, Cz, C3, C4, T7, T8, Pz, Oz).
5. **Biometric Performance Reporting**: Reports Accuracy, Equal Error Rate (EER), False Acceptance Rate (FAR), False Rejection Rate (FRR), False Positives (FP), and False Negatives (FN).


In [1]:
# Environment setup, reproducible random seeds, and imports
import os
import sys
import time
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
from scipy import signal
import mne
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve
from sklearn.svm import SVC
from sklearn.ensemble import ExtraTreesClassifier
import xgboost as xgb
import lightgbm as lgb

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Warmup CUDA
if torch.cuda.is_available():
    _ = torch.zeros(1, device=DEVICE)

print(f"Compute Device: {DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")
print(f"Random Seed   : {RANDOM_SEED}")


Compute Device: cuda (NVIDIA GeForce RTX 3050 Laptop GPU)
Random Seed   : 42


## Section 1: Preprocessing and Vectorized Spectral Feature Extraction

We implement the strictly spectral feature extraction pipeline.
For each electrode channel in every epoch, exactly 20 frequency-domain features are extracted from the Welch Power Spectral Density (PSD):
- 5 Absolute Band Powers: Delta (1-4 Hz), Theta (4-8 Hz), Alpha (8-13 Hz), Beta (13-30 Hz), Gamma (30-50 Hz)
- 1 Total Power: Broadband integral from 1.0 to 50.0 Hz
- 5 Relative Band Powers: Normalization against total broadband power
- 4 Spectral Ratios: Theta/Beta, Theta/Alpha, Alpha/Beta, Slow-to-Fast Ratio
- 5 Spectral Distribution Metrics: Spectral Centroid, Spread (Bandwidth), SEF50, SEF95, Peak Frequency
Statistical, time-domain, Hjorth, and entropy metrics are strictly excluded.


In [2]:
# High-performance vectorized spectral feature extractor
_trapezoid = getattr(np, "trapezoid", getattr(np, "trapz", None))

FREQ_BANDS = {
    "delta": (1.0, 4.0),
    "theta": (4.0, 8.0),
    "alpha": (8.0, 13.0),
    "beta":  (13.0, 30.0),
    "gamma": (30.0, 50.0)
}

def extract_spectral_features_tensor(data, sfreq=250.0):
    """
    Vectorized extraction of 20 spectral features per electrode channel.
    Input data: shape (n_epochs, n_channels, n_times)
    Output: shape (n_epochs, n_channels, 20)
    """
    n_epochs, n_channels, n_times = data.shape
    freqs, psd = signal.welch(data, fs=sfreq, nperseg=min(256, n_times), axis=-1)
    
    # 1. Absolute band powers (5)
    bp_list = []
    for (f_lo, f_hi) in FREQ_BANDS.values():
        mask = (freqs >= f_lo) & (freqs <= f_hi)
        p = _trapezoid(psd[:, :, mask], freqs[mask], axis=-1)
        bp_list.append(p)
    bp_arr = np.stack(bp_list, axis=-1)
    
    # 2. Total power (1-50 Hz) (1)
    tot_mask = (freqs >= 1.0) & (freqs <= 50.0)
    total_power = np.expand_dims(_trapezoid(psd[:, :, tot_mask], freqs[tot_mask], axis=-1), axis=-1) + 1e-12
    
    # 3. Relative band powers (5)
    rel_bp = bp_arr / total_power
    
    # 4. Spectral band power ratios (4)
    delta = bp_arr[..., 0] + 1e-12
    theta = bp_arr[..., 1] + 1e-12
    alpha = bp_arr[..., 2] + 1e-12
    beta  = bp_arr[..., 3] + 1e-12
    tbr = np.expand_dims(theta / beta, -1)
    tar = np.expand_dims(theta / alpha, -1)
    abr = np.expand_dims(alpha / beta, -1)
    slow_fast = np.expand_dims((delta + theta) / (alpha + beta), -1)
    ratios = np.concatenate([tbr, tar, abr, slow_fast], axis=-1)
    
    # 5. Spectral distribution & shape metrics (5)
    freqs_sub = freqs[tot_mask]
    psd_sub = psd[:, :, tot_mask]
    p_sum = psd_sub.sum(axis=-1, keepdims=True) + 1e-12
    centroid = (psd_sub * freqs_sub).sum(axis=-1, keepdims=True) / p_sum
    spread = np.sqrt(((freqs_sub - centroid)**2 * psd_sub).sum(axis=-1, keepdims=True) / p_sum)
    
    cum_psd = np.cumsum(psd_sub, axis=-1)
    cum_norm = cum_psd / (cum_psd[..., -1:] + 1e-12)
    idx50 = (cum_norm >= 0.5).argmax(axis=-1, keepdims=True)
    idx95 = (cum_norm >= 0.95).argmax(axis=-1, keepdims=True)
    sef50 = np.take_along_axis(np.broadcast_to(freqs_sub, idx50.shape[:-1] + (len(freqs_sub),)), idx50, axis=-1)
    sef95 = np.take_along_axis(np.broadcast_to(freqs_sub, idx95.shape[:-1] + (len(freqs_sub),)), idx95, axis=-1)
    
    idx_peak = psd_sub.argmax(axis=-1, keepdims=True)
    peak_freq = np.take_along_axis(np.broadcast_to(freqs_sub, idx_peak.shape[:-1] + (len(freqs_sub),)), idx_peak, axis=-1)
    spectral_dist = np.concatenate([centroid, spread, sef50, sef95, peak_freq], axis=-1)
    
    # Concatenate all 20 features per channel
    features = np.concatenate([bp_arr, total_power, rel_bp, ratios, spectral_dist], axis=-1)
    return features.astype(np.float32)

# Load dataset and extract spectral features with common epoch cap of 45
dataset_path = Path("../preprocessed_dataset")
if not dataset_path.exists():
    dataset_path = Path("preprocessed_dataset")

fif_files = sorted(list(dataset_path.glob("**/*_desc-preproc_epo.fif")))
EPOCH_CAP = 45

sample_ep = mne.read_epochs(str(fif_files[0]), preload=False, verbose=False)
ALL_CHANNELS = list(sample_ep.ch_names)
CHANNEL_TO_IDX = {ch: i for i, ch in enumerate(ALL_CHANNELS)}

REGIONS = {
    "Frontal": ["Fp1", "Fpz", "Fp2", "AF3", "AF4", "F7", "F5", "F3", "F1", "Fz", "F2", "F4", "F6", "F8"],
    "Central": ["FC5", "FC3", "FC1", "FCz", "FC2", "FC4", "FC6", "C5", "C3", "C1", "Cz", "C2", "C4", "C6"],
    "Temporal": ["T7", "T8", "TP7", "TP8", "CP5", "CP6"],
    "Parietal": ["CP3", "CP1", "CPz", "CP2", "CP4", "P7", "P5", "P3", "P1", "Pz", "P2", "P4", "P6", "P8"],
    "Occipital": ["PO7", "PO5", "PO3", "POz", "PO4", "PO6", "PO8", "O1", "Oz", "O2"],
    "ALL": ALL_CHANNELS
}

print(f"Scanning and caching spectral features for all {len(fif_files)} files (Capped at {EPOCH_CAP} epochs)...")
t0 = time.time()
dataset_cache = []

for fpath in fif_files:
    fname = fpath.name
    parts = fpath.parts
    sub = [p for p in parts if p.startswith("sub-")][0]
    ses = [p for p in parts if p.startswith("ses-")][0]
    task = "CE" if "_task-CE_" in fname else "OE"
    
    ep = mne.read_epochs(str(fpath), preload=True, verbose=False)
    raw_data = ep.get_data()[:EPOCH_CAP]
    sfreq = ep.info['sfreq']
    
    feats = extract_spectral_features_tensor(raw_data, sfreq=sfreq)
    
    dataset_cache.append({
        "subject": sub,
        "session": ses,
        "task": task,
        "features": feats,
        "n_epochs": feats.shape[0]
    })

print(f"Feature caching completed in {time.time() - t0:.2f} seconds.")
print(f"Total files cached: {len(dataset_cache)} | Total epochs: {len(dataset_cache) * EPOCH_CAP}")


Scanning and caching spectral features for all 40 files (Capped at 45 epochs)...


Feature caching completed in 10.19 seconds.
Total files cached: 40 | Total epochs: 1800


## Section 2: Biometric Metric Engine and Model Architectures

We implement the rigorous biometric evaluation protocol:
- **Equal Error Rate (EER)**: Computed directly from the continuous decision score distributions by finding the threshold where $FAR = FRR$.
- **Operational Metrics at 0.5 Threshold**: Accuracy, FAR, FRR, FP, FN.

We instantiate all 8 advanced models:
1. **XGBoost**: Gradient boosted decision tree ensemble with regularization.
2. **LightGBM**: Fast leaf-wise tree boosting.
3. **SVM-RBF**: Support Vector Machine with Gaussian Radial Basis Function kernel.
4. **Extra Trees**: Randomized decision tree forest.
5. **Res-MLP**: Deep Residual Multi-Layer Perceptron with skip connections and LayerNorm.
6. **1D-CNN**: Multi-channel 1D Convolutional Neural Network with adaptive average pooling.
7. **BiLSTM-Attention**: Bidirectional LSTM with Channel Self-Attention.
8. **SpecFormer**: Multi-Head Self-Attention Transformer encoder.


In [3]:
# Biometric Metric Calculator
def evaluate_biometric_predictions(y_true, y_scores, threshold=0.5):
    """Compute standard biometric verification metrics: EER, FAR, FRR, FP, FN, Accuracy."""
    y_true = np.asarray(y_true).astype(int)
    y_scores = np.asarray(y_scores).astype(float)
    
    n_gen = int(np.sum(y_true == 1))
    n_imp = int(np.sum(y_true == 0))
    
    fpr, tpr, thresholds = roc_curve(y_true, y_scores)
    fnr = 1.0 - tpr
    eer_idx = np.nanargmin(np.abs(fpr - fnr))
    eer = float((fpr[eer_idx] + fnr[eer_idx]) / 2.0)
    
    y_pred = (y_scores >= threshold).astype(int)
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    
    accuracy = float((tp + tn) / (n_gen + n_imp)) if (n_gen + n_imp) > 0 else 0.0
    far = float(fp / n_imp) if n_imp > 0 else 0.0
    frr = float(fn / n_gen) if n_gen > 0 else 0.0
    
    return {
        "N_train": 0,
        "N_genuine_test": n_gen,
        "N_impostor_test": n_imp,
        "Accuracy": accuracy,
        "EER": eer,
        "FAR": far,
        "FRR": frr,
        "FP": fp,
        "FN": fn
    }

# Deep Learning Architectures in PyTorch
class ResMLPNet(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.fc1 = nn.Linear(in_features, 64)
        self.relu = nn.ReLU()
        self.res1 = nn.Linear(64, 64)
        self.drop = nn.Dropout(0.1)
        self.head = nn.Linear(64, 1)
    def forward(self, x):
        h = self.relu(self.fc1(x))
        h = h + self.drop(self.relu(self.res1(h)))
        return self.head(h).view(-1)

class CNN1DNet(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=5, padding=2),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(4),
            nn.Flatten(),
            nn.Linear(32 * 4, 1)
        )
    def forward(self, x):
        x = x.unsqueeze(1)
        return self.net(x).view(-1)

class BiLSTMAttentionNet(nn.Module):
    def __init__(self, in_features, feat_dim=20):
        super().__init__()
        self.feat_dim = feat_dim
        dim = feat_dim if (in_features % feat_dim == 0 and in_features >= feat_dim) else in_features
        self.is_multichannel = (in_features % feat_dim == 0 and in_features >= feat_dim)
        self.lstm = nn.LSTM(dim if self.is_multichannel else 1, 16, batch_first=True, bidirectional=True)
        self.att = nn.Linear(32, 1)
        self.fc = nn.Linear(32, 1)
    def forward(self, x):
        if self.is_multichannel:
            x_seq = x.view(x.shape[0], -1, self.feat_dim)
        else:
            x_seq = x.unsqueeze(-1)
        out, _ = self.lstm(x_seq)
        weights = torch.softmax(self.att(out), dim=1)
        ctx = torch.sum(out * weights, dim=1)
        return self.fc(ctx).view(-1)

class SpecFormerNet(nn.Module):
    def __init__(self, in_features, d_model=24, nhead=3):
        super().__init__()
        if in_features % 20 == 0 and in_features >= 20:
            self.tok_proj = nn.Linear(20, d_model)
            self.multi_tok = True
        else:
            self.tok_proj = nn.Linear(1, d_model)
            self.multi_tok = False
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=48, batch_first=True, dropout=0.1)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=1)
        self.fc = nn.Linear(d_model, 1)
    def forward(self, x):
        if self.multi_tok:
            x_seq = x.view(x.shape[0], -1, 20)
        else:
            x_seq = x.unsqueeze(-1)
        h = self.tok_proj(x_seq)
        enc = self.encoder(h)
        return self.fc(enc.mean(dim=1)).view(-1)

# High-Performance PyTorch GPU Classifier Wrapper
class PyTorchClassifierWrapper:
    def __init__(self, net_cls, epochs=15, lr=3e-3, random_state=42):
        self.net_cls = net_cls
        self.epochs = epochs
        self.lr = lr
        self.random_state = random_state
        self.model = None

    def fit(self, X, y):
        torch.manual_seed(self.random_state)
        in_dim = X.shape[1]
        self.model = self.net_cls(in_dim).to(DEVICE)
        self.model.train()
        optimizer = torch.optim.Adam(self.model.parameters(), lr=self.lr)
        criterion = nn.BCEWithLogitsLoss()
        
        X_t = torch.tensor(X, dtype=torch.float32, device=DEVICE)
        y_t = torch.tensor(y, dtype=torch.float32, device=DEVICE)
        
        for _ in range(self.epochs):
            optimizer.zero_grad()
            out = self.model(X_t)
            loss = criterion(out, y_t)
            loss.backward()
            optimizer.step()
        return self

    def predict_proba(self, X):
        self.model.eval()
        X_t = torch.tensor(X, dtype=torch.float32, device=DEVICE)
        with torch.no_grad():
            logits = self.model(X_t)
            probs = torch.sigmoid(logits).cpu().numpy().flatten()
        return np.column_stack([1.0 - probs, probs])

def get_model_pipeline(model_name, in_features):
    if model_name == "XGBoost":
        return xgb.XGBClassifier(
            n_estimators=60, max_depth=4, learning_rate=0.1,
            eval_metric="logloss", random_state=RANDOM_SEED, n_jobs=-1
        )
    elif model_name == "LightGBM":
        return lgb.LGBMClassifier(
            n_estimators=60, max_depth=4, num_leaves=15, learning_rate=0.1,
            random_state=RANDOM_SEED, n_jobs=-1, verbose=-1
        )
    elif model_name == "SVM_RBF":
        return SVC(C=10.0, kernel="rbf", gamma="scale", probability=True, random_state=RANDOM_SEED)
    elif model_name == "ExtraTrees":
        return ExtraTreesClassifier(n_estimators=60, max_depth=None, random_state=RANDOM_SEED, n_jobs=-1)
    elif model_name == "Res-MLP":
        return PyTorchClassifierWrapper(ResMLPNet, epochs=15, lr=3e-3, random_state=RANDOM_SEED)
    elif model_name == "1D-CNN":
        return PyTorchClassifierWrapper(CNN1DNet, epochs=15, lr=3e-3, random_state=RANDOM_SEED)
    elif model_name == "BiLSTM-Attention":
        return PyTorchClassifierWrapper(BiLSTMAttentionNet, epochs=15, lr=3e-3, random_state=RANDOM_SEED)
    elif model_name == "SpecFormer":
        return PyTorchClassifierWrapper(SpecFormerNet, epochs=15, lr=3e-3, random_state=RANDOM_SEED)
    else:
        raise ValueError(f"Unknown model: {model_name}")

ALL_MODEL_NAMES = [
    "XGBoost", "LightGBM", "SVM_RBF", "ExtraTrees",
    "Res-MLP", "1D-CNN", "BiLSTM-Attention", "SpecFormer"
]
print("All 8 advanced model architectures verified and registered.")


All 8 advanced model architectures verified and registered.


## Section 3: Brain Region-Wise Model Evaluation

We evaluate all eight models across the five predefined brain regions:
- **Frontal** (14 channels $\times$ 20 features = 280 features)
- **Central** (14 channels $\times$ 20 features = 280 features)
- **Temporal** (6 channels $\times$ 20 features = 120 features)
- **Parietal** (14 channels $\times$ 20 features = 280 features)
- **Occipital** (10 channels $\times$ 20 features = 200 features)
- **ALL** (58 channels $\times$ 20 features = 1,160 features)

### 80:20 Impostor Testing Protocol:
1. Split genuine samples 80% train (288) / 20% test (72).
2. Balance with exactly 72 impostor test samples from other subjects' test sets (zero leakage).
3. Evaluate on balanced test set and report standard biometric metrics.


In [4]:
# Execute Region-Wise Biometric Evaluation
subjects = sorted(list(set(d['subject'] for d in dataset_cache)))

def slice_cached_features(channel_names):
    ch_indices = [CHANNEL_TO_IDX[ch] for ch in channel_names]
    X_by_subject = {s: [] for s in subjects}
    for item in dataset_cache:
        sub = item["subject"]
        sub_feats = item["features"][:, ch_indices, :].reshape(item["n_epochs"], -1)
        X_by_subject[sub].append(sub_feats)
    for sub in subjects:
        X_by_subject[sub] = np.vstack(X_by_subject[sub])
    return X_by_subject

region_results_list = []
print(f"Beginning Region-Wise Evaluation across {len(REGIONS)} configurations and {len(ALL_MODEL_NAMES)} models...\n")

for region_name, ch_list in REGIONS.items():
    print(f"--- Evaluating Region: {region_name} ({len(ch_list)} channels, {len(ch_list)*20} spectral features) ---")
    X_by_sub = slice_cached_features(ch_list)
    model_metrics = {m: [] for m in ALL_MODEL_NAMES}
    
    for target_sub in subjects:
        X_gen = X_by_sub[target_sub]
        n_gen_total = len(X_gen)
        n_gen_tr = int(n_gen_total * 0.8)
        n_gen_te = n_gen_total - n_gen_tr
        
        X_gen_tr = X_gen[:n_gen_tr]
        X_gen_te = X_gen[n_gen_tr:]
        
        other_subs = [s for s in subjects if s != target_sub]
        n_imp_tr_per_sub = n_gen_tr // len(other_subs)
        n_imp_te_per_sub = n_gen_te // len(other_subs)
        
        X_imp_tr_list = []
        X_imp_te_list = []
        for o_sub in other_subs:
            X_o = X_by_sub[o_sub]
            n_o_tr = int(len(X_o) * 0.8)
            X_imp_tr_list.append(X_o[:n_o_tr][:n_imp_tr_per_sub])
            X_imp_te_list.append(X_o[n_o_tr:][:n_imp_te_per_sub])
            
        X_train = np.vstack([X_gen_tr, np.vstack(X_imp_tr_list)])
        y_train = np.concatenate([np.ones(len(X_gen_tr)), np.zeros(len(np.vstack(X_imp_tr_list)))])
        
        X_test = np.vstack([X_gen_te, np.vstack(X_imp_te_list)])
        y_test = np.concatenate([np.ones(len(X_gen_te)), np.zeros(len(np.vstack(X_imp_te_list)))])
        
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        for model_name in ALL_MODEL_NAMES:
            clf = get_model_pipeline(model_name, in_features=X_train_scaled.shape[1])
            clf.fit(X_train_scaled, y_train)
            probs = clf.predict_proba(X_test_scaled)[:, 1]
            
            res = evaluate_biometric_predictions(y_test, probs)
            res["N_train"] = len(X_train)
            model_metrics[model_name].append(res)
            
    for model_name, metric_list in model_metrics.items():
        avg_res = {
            "Region": region_name,
            "Model": model_name,
            "N_train": int(np.mean([m["N_train"] for m in metric_list])),
            "N_genuine_test": int(np.mean([m["N_genuine_test"] for m in metric_list])),
            "N_impostor_test": int(np.mean([m["N_impostor_test"] for m in metric_list])),
            "Accuracy": float(np.mean([m["Accuracy"] for m in metric_list])),
            "EER": float(np.mean([m["EER"] for m in metric_list])),
            "FAR": float(np.mean([m["FAR"] for m in metric_list])),
            "FRR": float(np.mean([m["FRR"] for m in metric_list])),
            "FP": float(np.mean([m["FP"] for m in metric_list])),
            "FN": float(np.mean([m["FN"] for m in metric_list]))
        }
        region_results_list.append(avg_res)

df_region_results = pd.DataFrame(region_results_list)
print("\n" + "=" * 110)
print("REGION-WISE BIOMETRIC EVALUATION RESULTS (AVERAGED ACROSS ALL SUBJECTS):")
print("=" * 110)
format_dict = {
    "Accuracy": "{:.4f}",
    "EER": "{:.4f}",
    "FAR": "{:.4f}",
    "FRR": "{:.4f}",
    "FP": "{:.1f}",
    "FN": "{:.1f}"
}
display(df_region_results.style.format(format_dict))


Beginning Region-Wise Evaluation across 6 configurations and 8 models...

--- Evaluating Region: Frontal (14 channels, 280 spectral features) ---


--- Evaluating Region: Central (14 channels, 280 spectral features) ---


--- Evaluating Region: Temporal (6 channels, 120 spectral features) ---


--- Evaluating Region: Parietal (14 channels, 280 spectral features) ---


--- Evaluating Region: Occipital (10 channels, 200 spectral features) ---


--- Evaluating Region: ALL (58 channels, 1160 spectral features) ---



REGION-WISE BIOMETRIC EVALUATION RESULTS (AVERAGED ACROSS ALL SUBJECTS):


,Region,Model,N_train,N_genuine_test,N_impostor_test,Accuracy,EER,FAR,FRR,FP,FN
0,Frontal,XGBoost,576,72,72,0.8750,0.1333,0.1806,0.0694,13.0,5.0
1,Frontal,LightGBM,576,72,72,0.8806,0.1264,0.1750,0.0639,12.6,4.6
2,Frontal,SVM_RBF,576,72,72,0.8750,0.0903,0.1750,0.0750,12.6,5.4
3,Frontal,ExtraTrees,576,72,72,0.9083,0.1014,0.1500,0.0333,10.8,2.4
4,Frontal,Res-MLP,576,72,72,0.8722,0.1000,0.1694,0.0861,12.2,6.2
5,Frontal,1D-CNN,576,72,72,0.6639,0.2986,0.3306,0.3417,23.8,24.6
6,Frontal,BiLSTM-Attention,576,72,72,0.7472,0.2889,0.2667,0.2389,19.2,17.2
7,Frontal,SpecFormer,576,72,72,0.7944,0.1875,0.2611,0.1500,18.8,10.8
8,Central,XGBoost,576,72,72,0.9181,0.0681,0.1250,0.0389,9.0,2.8
9,Central,LightGBM,576,72,72,0.9139,0.0833,0.1500,0.0222,10.8,1.6


## Section 4: Individual Electrode-Wise Model Evaluation

We evaluate the discriminative power of individual electrode channels across key sites spanning all lobes:
- **Frontal**: Fp1, Fz
- **Central**: FCz, Cz, C3, C4
- **Temporal**: T7, T8
- **Parietal**: Pz
- **Occipital**: Oz

Each trial contains exactly 20 spectral-domain features.
All 8 models are evaluated under the identical 80:20 impostor testing protocol.


In [5]:
# Execute Individual Electrode Biometric Evaluation
REPRESENTATIVE_ELECTRODES = ["Fp1", "Fz", "FCz", "Cz", "C3", "C4", "T7", "T8", "Pz", "Oz"]

electrode_results_list = []
print(f"Beginning Electrode-Wise Evaluation across {len(REPRESENTATIVE_ELECTRODES)} electrodes and {len(ALL_MODEL_NAMES)} models...\n")

for elec in REPRESENTATIVE_ELECTRODES:
    X_by_sub = slice_cached_features([elec])
    model_metrics = {m: [] for m in ALL_MODEL_NAMES}
    
    for target_sub in subjects:
        X_gen = X_by_sub[target_sub]
        n_gen_tr = int(len(X_gen) * 0.8)
        n_gen_te = len(X_gen) - n_gen_tr
        
        X_gen_tr = X_gen[:n_gen_tr]
        X_gen_te = X_gen[n_gen_tr:]
        
        other_subs = [s for s in subjects if s != target_sub]
        n_imp_tr_per_sub = n_gen_tr // len(other_subs)
        n_imp_te_per_sub = n_gen_te // len(other_subs)
        
        X_imp_tr_list = []
        X_imp_te_list = []
        for o_sub in other_subs:
            X_o = X_by_sub[o_sub]
            n_o_tr = int(len(X_o) * 0.8)
            X_imp_tr_list.append(X_o[:n_o_tr][:n_imp_tr_per_sub])
            X_imp_te_list.append(X_o[n_o_tr:][:n_imp_te_per_sub])
            
        X_train = np.vstack([X_gen_tr, np.vstack(X_imp_tr_list)])
        y_train = np.concatenate([np.ones(len(X_gen_tr)), np.zeros(len(np.vstack(X_imp_tr_list)))])
        
        X_test = np.vstack([X_gen_te, np.vstack(X_imp_te_list)])
        y_test = np.concatenate([np.ones(len(X_gen_te)), np.zeros(len(np.vstack(X_imp_te_list)))])
        
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        for model_name in ALL_MODEL_NAMES:
            clf = get_model_pipeline(model_name, in_features=X_train_scaled.shape[1])
            clf.fit(X_train_scaled, y_train)
            probs = clf.predict_proba(X_test_scaled)[:, 1]
            res = evaluate_biometric_predictions(y_test, probs)
            res["N_train"] = len(X_train)
            model_metrics[model_name].append(res)
            
    for model_name, metric_list in model_metrics.items():
        avg_res = {
            "Electrode": elec,
            "Model": model_name,
            "N_train": int(np.mean([m["N_train"] for m in metric_list])),
            "N_genuine_test": int(np.mean([m["N_genuine_test"] for m in metric_list])),
            "N_impostor_test": int(np.mean([m["N_impostor_test"] for m in metric_list])),
            "Accuracy": float(np.mean([m["Accuracy"] for m in metric_list])),
            "EER": float(np.mean([m["EER"] for m in metric_list])),
            "FAR": float(np.mean([m["FAR"] for m in metric_list])),
            "FRR": float(np.mean([m["FRR"] for m in metric_list])),
            "FP": float(np.mean([m["FP"] for m in metric_list])),
            "FN": float(np.mean([m["FN"] for m in metric_list]))
        }
        electrode_results_list.append(avg_res)

df_electrode_results = pd.DataFrame(electrode_results_list)
print("\n" + "=" * 110)
print("ELECTRODE-WISE BIOMETRIC EVALUATION SUMMARY (TOP 15 CONFIGURATIONS BY LOWEST EER):")
print("=" * 110)
df_top_electrodes = df_electrode_results.sort_values(by=["EER", "Accuracy"], ascending=[True, False]).head(15)
display(df_top_electrodes.style.format(format_dict))


Beginning Electrode-Wise Evaluation across 10 electrodes and 8 models...




ELECTRODE-WISE BIOMETRIC EVALUATION SUMMARY (TOP 15 CONFIGURATIONS BY LOWEST EER):


,Electrode,Model,N_train,N_genuine_test,N_impostor_test,Accuracy,EER,FAR,FRR,FP,FN
1,Fp1,LightGBM,576,72,72,0.8903,0.1014,0.1083,0.1111,7.8,8.0
2,Fp1,SVM_RBF,576,72,72,0.8708,0.1139,0.1361,0.1222,9.8,8.8
3,Fp1,ExtraTrees,576,72,72,0.8847,0.1194,0.1194,0.1111,8.6,8.0
0,Fp1,XGBoost,576,72,72,0.8722,0.1236,0.1278,0.1278,9.2,9.2
8,Fz,XGBoost,576,72,72,0.8722,0.1250,0.1778,0.0778,12.8,5.6
10,Fz,SVM_RBF,576,72,72,0.8486,0.1292,0.2167,0.0861,15.6,6.2
17,FCz,LightGBM,576,72,72,0.8583,0.1306,0.1944,0.0889,14.0,6.4
9,Fz,LightGBM,576,72,72,0.8764,0.1319,0.1806,0.0667,13.0,4.8
16,FCz,XGBoost,576,72,72,0.8569,0.1319,0.1861,0.1000,13.4,7.2
66,Pz,SVM_RBF,576,72,72,0.8403,0.1319,0.1778,0.1417,12.8,10.2


## Section 5: Comparative Analysis and Best-Performing Configurations

We analyze the relative biometric discriminability across brain regions and single electrodes.
We identify the best regional configuration and the best individual electrode for each model.


In [6]:
# Identify Best Performing Configurations
print("=" * 100)
print("BEST REGION PER MODEL (RANKED BY LOWEST EQUAL ERROR RATE):")
print("=" * 100)
best_regions = df_region_results.loc[df_region_results.groupby("Model")["EER"].idxmin()]
display(best_regions[["Model", "Region", "Accuracy", "EER", "FAR", "FRR", "FP", "FN"]].sort_values(by="EER").style.format(format_dict))

print("\n" + "=" * 100)
print("BEST INDIVIDUAL ELECTRODE PER MODEL (RANKED BY LOWEST EQUAL ERROR RATE):")
print("=" * 100)
best_electrodes = df_electrode_results.loc[df_electrode_results.groupby("Model")["EER"].idxmin()]
display(best_electrodes[["Model", "Electrode", "Accuracy", "EER", "FAR", "FRR", "FP", "FN"]].sort_values(by="EER").style.format(format_dict))

# Overall winner summary
best_overall_region = df_region_results.loc[df_region_results["EER"].idxmin()]
best_overall_elec = df_electrode_results.loc[df_electrode_results["EER"].idxmin()]

print("\n" + "=" * 80)
print("OVERALL BIOMETRIC IDENTIFICATION LEADERS:")
print("=" * 80)
print(f"Best Regional Configuration : {best_overall_region['Region']} with {best_overall_region['Model']} (EER: {best_overall_region['EER']:.4f}, Accuracy: {best_overall_region['Accuracy']:.4f})")
print(f"Best Single Electrode       : {best_overall_elec['Electrode']} with {best_overall_elec['Model']} (EER: {best_overall_elec['EER']:.4f}, Accuracy: {best_overall_elec['Accuracy']:.4f})")
print("Notebook 2 evaluation completed successfully.")


BEST REGION PER MODEL (RANKED BY LOWEST EQUAL ERROR RATE):


,Model,Region,Accuracy,EER,FAR,FRR,FP,FN
44,Res-MLP,ALL,0.9097,0.0583,0.0889,0.0917,6.4,6.6
11,ExtraTrees,Central,0.9125,0.0611,0.1361,0.0389,9.8,2.8
42,SVM_RBF,ALL,0.9486,0.0639,0.0750,0.0278,5.4,2.0
41,LightGBM,ALL,0.9264,0.0639,0.0861,0.0611,6.2,4.4
40,XGBoost,ALL,0.9181,0.0653,0.0806,0.0833,5.8,6.0
7,SpecFormer,Frontal,0.7944,0.1875,0.2611,0.1500,18.8,10.8
30,BiLSTM-Attention,Parietal,0.7542,0.2458,0.2722,0.2194,19.6,15.8
29,1D-CNN,Parietal,0.6500,0.2875,0.3806,0.3194,27.4,23.0



BEST INDIVIDUAL ELECTRODE PER MODEL (RANKED BY LOWEST EQUAL ERROR RATE):


,Model,Electrode,Accuracy,EER,FAR,FRR,FP,FN
1,LightGBM,Fp1,0.8903,0.1014,0.1083,0.1111,7.8,8.0
2,SVM_RBF,Fp1,0.8708,0.1139,0.1361,0.1222,9.8,8.8
3,ExtraTrees,Fp1,0.8847,0.1194,0.1194,0.1111,8.6,8.0
0,XGBoost,Fp1,0.8722,0.1236,0.1278,0.1278,9.2,9.2
15,SpecFormer,Fz,0.8167,0.1625,0.2444,0.1222,17.6,8.8
20,Res-MLP,FCz,0.7903,0.2014,0.2389,0.1806,17.2,13.0
45,1D-CNN,C4,0.7792,0.2125,0.2806,0.1611,20.2,11.6
46,BiLSTM-Attention,C4,0.7639,0.2514,0.2472,0.2250,17.8,16.2



OVERALL BIOMETRIC IDENTIFICATION LEADERS:
Best Regional Configuration : ALL with Res-MLP (EER: 0.0583, Accuracy: 0.9097)
Best Single Electrode       : Fp1 with LightGBM (EER: 0.1014, Accuracy: 0.8903)
Notebook 2 evaluation completed successfully.
